In [1]:
# === SETUP: Run this first! ===
import os
import sys

# Change to project root and add to Python path
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)  # Goes up one level from 'notebooks/'
os.chdir(project_root)
sys.path.insert(0, project_root)

print(f"Project root: {project_root}")
print(f"tsnn module path: {os.path.join(project_root, 'tsnn')}")

Project root: /Users/gremy/Code/TSNN-1
tsnn module path: /Users/gremy/Code/TSNN-1/tsnn


In [2]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.linear_model import LassoCV, RidgeCV, LinearRegression
import torch
from torch.utils.data import Dataset, random_split
from torch.utils.data import DataLoader
import importlib
import sys
#sys.path.append('/Users/cyrilgarcia/notebooks/tsnn/')

import tsnn

from tsnn.generators import generators
from tsnn.benchmarks import benchmark_comparison, ml_benchmarks, torch_benchmarks
from tsnn import utils
import torch.nn.functional as F
import math
from typing import Optional
from tsnn.tstorch import transformers



plt.style.use('ggplot')

In [3]:
from dataclasses import dataclass
from torch import nn
device = 'mps'

In [4]:
from typing import Dict

In [5]:
from tsnn.tstorch import models

In [6]:
from tsnn.tstorch.models import GlobalMLP, BiDimensionalMLP, OneDimensionalTransformer, CustomBiDimensionalTransformer
from sklearn.ensemble import HistGradientBoostingRegressor



# Summary

In [7]:
# In this notebook we will run the experiments to generate the figures for the paper.

# Test dataset

In [8]:
# We work with the following data.

In [9]:
# Global parameters
T_max = 5000
N1 = 10
F1 = 20
T1 = 5 # This parameter will be the n_rolling

print(T_max, N1, F1, T1)

5000 10 20 5


In [10]:
def generate_synthetic_datasets(
    num_time_steps: int = 3000,
    num_time_series: int = 10,
    num_features: int = 10,
    low_corr: float = 0.1,
    high_corr: float = 0.2,
    pct_zero_corr: float = 0.5,
) -> Dict[str, "generators.Generator"]:
    """
    Generates 5 synthetic multivariate time series datasets with different
    types of cross-series dependencies.

    Returns
    -------
    dict
        Keys: "d_lin", "d_cond", "d_shift", "d_cs", "d_cs_shift", "d_all"
        Values: generators.Generator objects (already with .train and .test)
    """
    dic_data = {}

    # Helper to avoid repeating the same 10 lines
    def make_gen(split_conditional=0.0,
                 split_shift=0.0,
                 split_seasonal=0.0,
                 split_cs=0.0,
                 split_cs_shift=0.0):
        gen = generators.Generator(num_time_steps, num_time_series, num_features)
        gen.generate_dataset(
            pct_zero_corr=pct_zero_corr,
            split_conditional=split_conditional,
            split_shift=split_shift,
            split_seasonal=split_seasonal,
            split_cs=split_cs,
            split_cs_shift=split_cs_shift,
            low_corr=low_corr,
            high_corr=high_corr,
        )
        return gen

    dic_data["d_lin"] = make_gen()

    # 1. Pure conditional (causal) dependence
    dic_data["d_cond"] = make_gen(split_conditional=1.0)

    # 2. Pure lagged (time-shifted) dependence
    dic_data["d_shift"] = make_gen(split_shift=1.0)

    # 3. Pure contemporaneous cross-sectional correlation
    dic_data["d_cs"] = make_gen(split_cs=1.0)

    # 4. Contemporaneous + lagged cross-series
    dic_data["d_cs_shift"] = make_gen(split_cs_shift=1.0)

    # 5. Equal mix of all four mechanisms
    dic_data["d_all"] = make_gen(
        split_conditional=0.2,
        split_shift=0.2,
        split_cs=0.2,
        split_cs_shift=0.2,
    )

    return dic_data

In [11]:
# list_low_corr = [0.01, 0.025, 0.05, 0.1]
# list_high_corr = [2*x for x in list_low_corr]

list_low_corr = [0.01, 0.03, 0.05, 0.1, 0.3, 0.5]
list_high_corr = list_low_corr

dic_data = {}

for i in range(len(list_low_corr)):
    name = "correl" + str(list_low_corr[i])
    dic_data[name] = generate_synthetic_datasets(num_time_steps=T_max, num_time_series=N1, num_features=F1, low_corr=list_low_corr[i], high_corr=list_high_corr[i])
    

In [12]:
dic_data.keys()

dict_keys(['correl0.01', 'correl0.03', 'correl0.05', 'correl0.1', 'correl0.3', 'correl0.5'])

In [13]:
effects = list(dic_data['correl0.1'].keys())
print(effects)

['d_lin', 'd_cond', 'd_shift', 'd_cs', 'd_cs_shift', 'd_all']


In [14]:
# We will fix the above dataset for now.

In [15]:
def causal_mask(b, h, q_idx, kv_idx):
    return q_idx >= kv_idx

def build_attention_mask(mask_fn, seq_len, device="cpu"):
    q_idx = torch.arange(seq_len, device=device)
    kv_idx = torch.arange(seq_len, device=device)
    b = torch.zeros(1, device=device)
    h = torch.zeros(1, device=device)
    mask_bool = mask_fn(b, h, q_idx[:, None], kv_idx[None, :])  # (seq_len, seq_len)
    return mask_bool

mask = causal_mask
mask = build_attention_mask(mask, T1, device=device)
def custom_mask_mod(b, h, q_idx, kv_idx):
    return mask[q_idx, kv_idx]

# List of models

In [16]:
# Let's list here all the models we wish to test on all the data.

In [17]:
MLP_global = GlobalMLP(N1, F1, T1, dropout=0.2).to(device)

MLP_2D = BiDimensionalMLP(N1, F1, T1, dropout=0.2).to(device)

trans_1D_T4 = OneDimensionalTransformer(N1, F1, T1, mask=mask, attn_direction="T",  num_attn_layers=4,
                                        dropout=0.2, roll_y=True).to(device)
#Note: using the MLP compression seems very bad..

trans_2D_TCTC = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTC',
                                                                    dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=None, 
                                                                           roll_y=True).to(device)

trans_2D_TCTCTCTC = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTCTCTC',
                                                                    dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=None, 
                                                                           roll_y=True).to(device)


In [18]:
dic_models = {'MLP_global':MLP_global, 'MLP_2D':MLP_2D, "trans_1D_T4":trans_1D_T4, "trans_2D_TCTC":trans_2D_TCTC, "trans_2D_TCTCTCTC":trans_2D_TCTCTCTC}

In [19]:
trans_2D_TCTC_rollfalse = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTC',
                                                                    dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=None, 
                                                                           roll_y=False).to(device)

trans_2D_TCTCTCTC_rollfalse = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTCTCTC',
                                                                    dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=None, 
                                                                           roll_y=False).to(device)


In [20]:
dic_models_rollfalse = {"trans_2D_TCTC":trans_2D_TCTC_rollfalse, "trans_2D_TCTCTCTC":trans_2D_TCTCTCTC_rollfalse}

In [21]:
# For each model we also need to specify the option we will use to fit them.

In [22]:
dic_data['correl0.1']

{'d_lin': <tsnn.generators.generators.Generator at 0x16a4c3610>,
 'd_cond': <tsnn.generators.generators.Generator at 0x17398b890>,
 'd_shift': <tsnn.generators.generators.Generator at 0x173953b90>,
 'd_cs': <tsnn.generators.generators.Generator at 0x1739abd90>,
 'd_cs_shift': <tsnn.generators.generators.Generator at 0x1739ab350>,
 'd_all': <tsnn.generators.generators.Generator at 0x173953d10>}

In [23]:
dic_data.keys()

dict_keys(['correl0.01', 'correl0.03', 'correl0.05', 'correl0.1', 'correl0.3', 'correl0.5'])

In [24]:
list(dic_models.keys())

['MLP_global', 'MLP_2D', 'trans_1D_T4', 'trans_2D_TCTC', 'trans_2D_TCTCTCTC']

# Function to create table for an effect

In [25]:
# We give the function that creates for a given effect the table testing all models and all noise level.

In [26]:
def run_models(effect1):

    # Storage
    records_train = []
    records_test  = []

    for noise_level in dic_data.keys():
        
        z = dic_data[noise_level][effect1]
        z.get_dataloader(n_rolling=T1)

        lasso_full = ml_benchmarks.CustomBenchmarkRolling(LassoCV(alphas=np.linspace(start=1e-4, stop=1e-2, num=30), 
                                                                verbose=False, max_iter=2000, selection='random', n_jobs=5, cv=3))
        lasso_full.fit(z.train)
        boost_model = ml_benchmarks.CustomBenchmarkRolling(HistGradientBoostingRegressor(max_depth=6, max_iter=500, validation_fraction=0.3, n_iter_no_change=100))
        boost_model.fit(z.train)

        comp = benchmark_comparison.Comparator(models=[lasso_full, boost_model], model_names=['lasso_full', 'boosting'])
        corr_train = comp.correl(z, mode="train", return_values=True)
        corr_test  = comp.correl(z, mode="test",  return_values=True)

        # Save results
        records_train.append({
            "noise_level": noise_level,
            "model": 'lasso_full',
            "train_corr_optimal": corr_train.loc['lasso_full', "optimal"]
        })
        records_test.append({
            "noise_level": noise_level,
            "model": 'lasso_full',
            "test_corr_optimal": corr_test.loc['lasso_full', "optimal"]
        })

        records_train.append({
            "noise_level": noise_level,
            "model": 'boosting',
            "train_corr_optimal": corr_train.loc['boosting', "optimal"]
        })
        records_test.append({
            "noise_level": noise_level,
            "model": 'boosting',
            "test_corr_optimal": corr_test.loc['boosting', "optimal"]
        })
        


        for model_key in dic_models.keys():                   
            print(f"Running → {noise_level} | {model_key}")

            z = dic_data[noise_level][effect1]
            if model_key in ["trans_1D_T4", "trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
                z.get_dataloader(n_rolling=T1, roll_y=True)
            else:
                z.get_dataloader(n_rolling=T1)

            if model_key in ["trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
                lr=0.001/2
            else:
                lr=0.001

            epochs = 20
            if noise_level in ['correl0.01', 'correl0.03']:
                epochs = 40

            model = dic_models[model_key]

            # Model
            if noise_level == 'correl0.01' and model_key in ["trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
                z.get_dataloader(n_rolling=T1)
                model = dic_models_rollfalse[model_key]
                epochs = 60
                lr=0.0001

            optimizer = torch.optim.Adam(model.parameters(), lr=lr)
            wrapper = torch_benchmarks.TorchWrapper(model, optimizer=optimizer)

            
            # Train
            wrapper.fit(z.train, test=z.test, epochs=epochs, plot=False)

            # Compare
            comp = benchmark_comparison.Comparator(models=[wrapper], model_names=["model1"])

            corr_train = comp.correl(z, mode="train", return_values=True)
            corr_test  = comp.correl(z, mode="test",  return_values=True)

            train_corr = corr_train.loc["model1", "optimal"]
            test_corr  = corr_test.loc["model1", "optimal"]

            # Save results
            records_train.append({
                "noise_level": noise_level,
                "model": model_key,
                "train_corr_optimal": train_corr
            })
            records_test.append({
                "noise_level": noise_level,
                "model": model_key,
                "test_corr_optimal": test_corr
            })

    # ————————————————————————————————————————
    # 3. Convert to nice DataFrames
    # ————————————————————————————————————————
    df_train = pd.DataFrame(records_train)
    df_test  = pd.DataFrame(records_test)

    # Pivot: rows = noise level, columns = effect type
    train_pivot = df_train.pivot(index="noise_level", columns="model", values="train_corr_optimal")
    test_pivot  = df_test.pivot(index="noise_level", columns="model", values="test_corr_optimal")

    col_order = ["lasso_full", "boosting"] + list(dic_models.keys())
    train_pivot = train_pivot[col_order]
    test_pivot  = test_pivot[col_order]

    return train_pivot, test_pivot

## Running on linear effect

In [27]:
table_train_lin0, table_test_lin0 = run_models('d_lin')

Running → correl0.01 | MLP_global


100%|██████████| 40/40 [00:06<00:00,  6.45it/s]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → correl0.01 | MLP_2D


100%|██████████| 40/40 [00:09<00:00,  4.25it/s]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → correl0.01 | trans_1D_T4


...

100%|██████████| 20/20 [00:08<00:00,  2.29it/s]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → correl0.5 | trans_2D_TCTC


100%|██████████| 20/20 [00:23<00:00,  1.19s/it]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → correl0.5 | trans_2D_TCTCTCTC


100%|██████████| 20/20 [00:44<00:00,  2.22s/it]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


In [28]:
display(table_train_lin0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,0.006,0.015,0.033,0.028,0.033,0.182,0.143
correl0.03,0.040,0.026,0.083,0.083,0.088,0.101,0.095
correl0.05,0.046,0.036,0.159,0.173,0.174,0.208,0.184
correl0.1,0.166,0.088,0.293,0.343,0.322,0.382,0.306
correl0.3,0.310,0.198,0.694,0.817,0.732,0.849,0.770
correl0.5,0.315,0.236,0.850,0.925,0.888,0.980,0.957


In [29]:
display(table_test_lin0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,-0.002,0.009,0.047,0.019,0.056,0.191,0.156
correl0.03,0.045,0.008,0.086,0.041,0.118,0.118,0.138
correl0.05,0.046,0.019,0.088,0.124,0.216,0.256,0.261
correl0.1,0.144,0.035,0.179,0.422,0.446,0.448,0.426
correl0.3,0.307,0.187,0.447,0.864,0.822,0.875,0.841
correl0.5,0.311,0.250,0.797,0.938,0.923,0.981,0.962


In [30]:
# Saving the data

table_train_lin0.to_csv('table_train_lin.csv', index=True)
table_test_lin0.to_csv('table_test_lin.csv', index=True)

In [31]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

## Running on conditional effect

In [32]:
table_train_cond0, table_test_cond0 = run_models('d_cond')

Running → correl0.01 | MLP_global


100%|██████████| 40/40 [00:06<00:00,  6.61it/s]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → correl0.01 | MLP_2D


100%|██████████| 40/40 [00:10<00:00,  3.89it/s]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → correl0.01 | trans_1D_T4


...

100%|██████████| 20/20 [00:08<00:00,  2.28it/s]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → correl0.5 | trans_2D_TCTC


100%|██████████| 20/20 [00:23<00:00,  1.19s/it]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → correl0.5 | trans_2D_TCTCTCTC


100%|██████████| 20/20 [00:44<00:00,  2.21s/it]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


In [33]:
display(table_train_cond0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,-0.001,0.011,0.034,0.028,0.035,0.062,0.062
correl0.03,0.023,0.034,0.090,0.086,0.092,0.096,0.092
correl0.05,0.014,0.037,0.145,0.123,0.149,0.166,0.154
correl0.1,0.035,0.080,0.297,0.243,0.302,0.338,0.314
correl0.3,0.078,0.178,0.661,0.548,0.677,0.740,0.704
correl0.5,0.102,0.221,0.796,0.653,0.830,0.896,0.871


In [34]:
display(table_test_cond0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,-0.002,-0.001,-0.001,-0.005,0.003,0.059,0.065
correl0.03,-0.013,-0.012,-0.008,0.002,-0.001,0.071,0.069
correl0.05,0.006,0.004,0.003,-0.003,0.011,0.137,0.109
correl0.1,-0.014,-0.004,-0.004,0.011,0.009,0.258,0.264
correl0.3,-0.011,-0.004,0.007,0.018,0.029,0.649,0.636
correl0.5,0.012,-0.010,0.002,0.048,0.065,0.845,0.833


In [35]:
# Saving the data

table_train_cond0.to_csv('table_train_cond.csv', index=True)
table_test_cond0.to_csv('table_test_cond.csv', index=True)

In [36]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

## Running on shift effect

In [37]:
table_train_shift0, table_test_shift0 = run_models('d_shift')

Running → correl0.01 | MLP_global


100%|██████████| 40/40 [00:06<00:00,  6.46it/s]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → correl0.01 | MLP_2D


100%|██████████| 40/40 [00:09<00:00,  4.28it/s]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → correl0.01 | trans_1D_T4


...

100%|██████████| 20/20 [00:08<00:00,  2.48it/s]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → correl0.5 | trans_2D_TCTC


100%|██████████| 20/20 [00:24<00:00,  1.20s/it]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → correl0.5 | trans_2D_TCTCTCTC


100%|██████████| 20/20 [00:44<00:00,  2.23s/it]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


In [38]:
display(table_train_shift0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,0.018,0.014,0.025,0.031,0.027,0.077,0.050
correl0.03,0.018,0.016,0.106,0.106,0.105,0.112,0.107
correl0.05,0.058,0.038,0.147,0.136,0.149,0.162,0.150
correl0.1,0.150,0.081,0.290,0.324,0.302,0.334,0.311
correl0.3,0.303,0.193,0.658,0.829,0.695,0.764,0.724
correl0.5,0.307,0.233,0.806,0.946,0.863,0.937,0.896


In [39]:
display(table_test_shift0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,0.020,0.017,0.032,0.024,-0.011,0.090,0.055
correl0.03,0.021,0.008,0.037,0.039,0.022,0.074,0.057
correl0.05,0.058,0.032,0.100,0.056,0.058,0.129,0.083
correl0.1,0.133,0.030,0.090,0.232,0.086,0.275,0.209
correl0.3,0.297,0.186,0.346,0.862,0.406,0.771,0.722
correl0.5,0.316,0.254,0.547,0.942,0.720,0.938,0.899


In [40]:
# Saving the data

table_train_shift0.to_csv('table_train_shift.csv', index=True)
table_test_shift0.to_csv('table_test_shift.csv', index=True)

In [41]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

## Running on the cs effect

In [42]:
table_train_cs0, table_test_cs0 = run_models('d_cs')

Running → correl0.01 | MLP_global


100%|██████████| 40/40 [00:06<00:00,  6.61it/s]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → correl0.01 | MLP_2D


100%|██████████| 40/40 [00:09<00:00,  4.34it/s]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → correl0.01 | trans_1D_T4


...

100%|██████████| 20/20 [00:08<00:00,  2.43it/s]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → correl0.5 | trans_2D_TCTC


100%|██████████| 20/20 [00:25<00:00,  1.26s/it]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → correl0.5 | trans_2D_TCTCTCTC


100%|██████████| 20/20 [00:47<00:00,  2.35s/it]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


In [43]:
display(table_train_cs0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,0.001,0.003,0.029,0.025,0.034,0.040,0.030
correl0.03,0.033,0.031,0.089,0.086,0.095,0.093,0.089
correl0.05,0.066,0.055,0.146,0.137,0.173,0.152,0.154
correl0.1,0.168,0.091,0.293,0.343,0.346,0.319,0.307
correl0.3,0.295,0.185,0.643,0.848,0.749,0.789,0.714
correl0.5,0.311,0.232,0.833,0.935,0.898,0.924,0.878


In [44]:
display(table_test_cs0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,0.008,-0.007,0.018,0.002,0.044,0.023,0.007
correl0.03,0.027,0.011,0.039,0.013,0.127,0.042,0.053
correl0.05,0.055,0.019,0.068,0.034,0.209,0.054,0.065
correl0.1,0.135,0.037,0.157,0.258,0.426,0.171,0.151
correl0.3,0.312,0.186,0.397,0.869,0.804,0.818,0.768
correl0.5,0.315,0.250,0.724,0.936,0.926,0.941,0.926


In [45]:
# Saving the data

table_train_cs0.to_csv('table_train_cs.csv', index=True)
table_test_cs0.to_csv('table_test_cs.csv', index=True)

In [46]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

## Running on the cs_shift

In [47]:
table_train_csshift0, table_test_csshift0 = run_models('d_cs_shift')

Running → correl0.01 | MLP_global


100%|██████████| 40/40 [00:06<00:00,  5.82it/s]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → correl0.01 | MLP_2D


100%|██████████| 40/40 [00:09<00:00,  4.13it/s]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → correl0.01 | trans_1D_T4


...

100%|██████████| 20/20 [00:08<00:00,  2.35it/s]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → correl0.5 | trans_2D_TCTC


100%|██████████| 20/20 [00:25<00:00,  1.26s/it]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → correl0.5 | trans_2D_TCTCTCTC


100%|██████████| 20/20 [00:46<00:00,  2.33s/it]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


In [48]:
display(table_train_csshift0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,0.001,-0.000,0.020,0.024,0.022,0.005,0.015
correl0.03,0.004,0.016,0.086,0.096,0.087,0.088,0.088
correl0.05,0.055,0.041,0.158,0.152,0.156,0.168,0.158
correl0.1,0.151,0.082,0.294,0.350,0.296,0.319,0.295
correl0.3,0.306,0.191,0.673,0.849,0.696,0.764,0.714
correl0.5,0.320,0.242,0.846,0.937,0.873,0.918,0.875


In [49]:
display(table_test_csshift0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,-0.008,0.000,0.013,0.029,0.017,-0.009,-0.002
correl0.03,0.013,0.006,0.070,0.054,0.038,0.047,0.043
correl0.05,0.044,0.010,0.077,0.088,0.044,0.066,0.028
correl0.1,0.127,0.046,0.172,0.298,0.095,0.213,0.090
correl0.3,0.303,0.165,0.481,0.870,0.398,0.744,0.567
correl0.5,0.307,0.243,0.739,0.938,0.798,0.919,0.857


In [50]:
# Saving the data

table_train_csshift0.to_csv('table_train_csshift.csv', index=True)
table_test_csshift0.to_csv('table_test_csshift.csv', index=True)

In [51]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

# Function to create table for all effects

In [ ]:
def run_models_all_effect(noise_level1):

    z = dic_data[noise_level1]["d_all"]
    z.get_dataloader(n_rolling=T1)

    lasso_full = ml_benchmarks.CustomBenchmarkRolling(LassoCV(alphas=np.linspace(start=1e-4, stop=1e-2, num=30), 
                                                            verbose=False, max_iter=2000, selection='random', n_jobs=5, cv=3))
    lasso_full.fit(z.train)
    boost_model = ml_benchmarks.CustomBenchmarkRolling(HistGradientBoostingRegressor(max_depth=6, max_iter=500, validation_fraction=0.3, n_iter_no_change=100))
    boost_model.fit(z.train)

    list_models = [lasso_full, boost_model]


    for model_key in dic_models.keys():                   
        print(f"Running → {noise_level1} | {model_key}")

        if model_key in ["trans_1D_T4", "trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
            z.get_dataloader(n_rolling=T1, roll_y=True)
        else:
            z.get_dataloader(n_rolling=T1)

        if model_key in ["trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
            lr=0.001/2
        else:
            lr=0.001

        epochs = 20
        if noise_level1 in ['noise0.01', 'noise0.03']:
            epochs = 40

        model = dic_models[model_key]

        # Model
        if noise_level1 == 'noise0.01' and model_key in ["trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
            z.get_dataloader(n_rolling=T1)
            model = dic_models_rollfalse[model_key]
            epochs = 60
            lr=0.0001

        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        wrapper = torch_benchmarks.TorchWrapper(model, optimizer=optimizer)

        
        # Train
        wrapper.fit(z.train, test=z.test, epochs=epochs, plot=False)

        list_models.append(wrapper)

    comp = benchmark_comparison.Comparator(models=list_models, model_names=["lasso_full", "boosting"] + list(dic_models.keys()))

    corr_train = comp.correl(z, mode="train", return_values=True)
    corr_test  = comp.correl(z, mode="test",  return_values=True)

    return corr_train, corr_test

    

In [46]:
a1, b1 = run_models_all_effect("correl0.1")

Running → correl0.1 | MLP_global


100%|██████████| 20/20 [00:03<00:00,  6.45it/s]


Running → correl0.1 | MLP_2D


100%|██████████| 20/20 [00:04<00:00,  4.23it/s]


Running → correl0.1 | trans_1D_T4


100%|██████████| 20/20 [00:08<00:00,  2.24it/s]


Running → correl0.1 | trans_2D_TCTC


100%|██████████| 20/20 [00:24<00:00,  1.22s/it]


Running → correl0.1 | trans_2D_TCTCTCTC


100%|██████████| 20/20 [00:44<00:00,  2.22s/it]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


In [48]:
display(a1.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.303,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.141,0.466,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.131,0.463,0.017,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.136,0.459,0.013,0.024,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.145,0.465,0.024,0.018,0.015,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.150,0.470,0.026,0.021,0.018,0.023,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.128,0.123,0.060,0.017,0.108,0.068,0.034,nan,nan,nan,nan,nan,nan
boosting,0.252,0.079,0.036,0.037,0.043,0.038,0.030,0.375,nan,nan,nan,nan,nan
MLP_global,0.983,0.300,0.139,0.130,0.136,0.142,0.151,0.128,0.249,nan,nan,nan,nan
MLP_2D,0.939,0.313,0.150,0.125,0.145,0.154,0.154,0.128,0.237,0.923,nan,nan,nan


In [49]:
display(b1.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.308,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.144,0.470,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.144,0.466,0.028,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.138,0.452,0.021,0.012,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.148,0.467,0.016,0.024,0.011,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.142,0.469,0.024,0.019,0.011,0.033,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.026,0.104,0.057,-0.004,0.101,0.062,0.027,nan,nan,nan,nan,nan,nan
boosting,-0.004,0.030,0.013,0.002,0.022,0.011,0.020,0.257,nan,nan,nan,nan,nan
MLP_global,0.074,0.259,0.149,0.023,0.124,0.129,0.175,0.083,0.032,nan,nan,nan,nan
MLP_2D,0.094,0.277,0.158,0.017,0.146,0.162,0.161,0.069,0.033,0.247,nan,nan,nan


# Testing sparsity

In [50]:
# Let's now test sparsity at different levels of noise for the different effects.